# Cert Prep — 02: DataFrame API

**Exam weight: ~35% (largest section)**

Topics covered:
- select, selectExpr, withColumn, withColumnRenamed, drop
- filter / where
- sort / orderBy
- distinct, dropDuplicates
- groupBy + agg functions
- joins (inner, left, right, full, cross, semi, anti)
- union, unionByName
- Missing data: isNull, isNotNull, fillna, dropna, replace
- Reading / writing with schema
- Repartition vs coalesce


In [ ]:
import os, time, pathlib
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *

MASTER = 'spark://spark-master:7077'

spark = (
    SparkSession.builder
    .appName('cert-prep')
    .master(MASTER)
    .config('spark.executor.memory', '2g')
    .config('spark.driver.memory',   '1g')
    .config('spark.executor.cores',  '2')
    .config('spark.sql.warehouse.dir', '/workspace/data/training_warehouse')
    .getOrCreate()
)
spark.sparkContext.setLogLevel('WARN')
print(f"Spark {spark.version}")

# Sample dataset used throughout
data = [
    (1, "Alice",  "Engineering", 95000.0, "2021-03-15", 4),
    (2, "Bob",    "Marketing",   72000.0, "2020-07-01", 3),
    (3, "Carol",  "Engineering", 105000.0,"2019-11-20", 5),
    (4, "Dave",   "Marketing",   68000.0, "2022-01-10", 2),
    (5, "Eve",    "Engineering", 88000.0, "2021-09-05", 4),
    (6, "Frank",  "HR",          61000.0, "2020-04-18", 3),
    (7, "Grace",  "HR",          None,    "2023-02-28", 1),
    (8, "Hank",   "Engineering", 112000.0,"2018-06-12", 5),
    (9, "Iris",   "Marketing",   None,    "2022-11-03", 2),
    (10,"Jack",   "HR",          59000.0, "2023-08-01", 1),
]
schema = StructType([
    StructField("id",         IntegerType(),  False),
    StructField("name",       StringType(),   True),
    StructField("dept",       StringType(),   True),
    StructField("salary",     DoubleType(),   True),
    StructField("hire_date",  StringType(),   True),
    StructField("perf_score", IntegerType(),  True),
])
df = spark.createDataFrame(data, schema)
df.cache().count()
print("Dataset ready")
df.show()

In [ ]:
# ── select, withColumn, withColumnRenamed, drop ──────────────────────────────
print("=== Column Operations ===")

# select — pick columns, apply expressions
df.select("name", "dept", F.col("salary") * 1.1).show(3)

# selectExpr — SQL expressions as strings
df.selectExpr("name", "dept", "salary * 1.1 as salary_adjusted").show(3)

# withColumn — add or replace a column
df2 = df.withColumn("salary_k", F.round(F.col("salary") / 1000, 1))
df2.select("name","salary","salary_k").show(3)

# withColumnRenamed
df3 = df.withColumnRenamed("perf_score", "score")
print("Renamed columns:", df3.columns)

# drop
df4 = df.drop("hire_date", "perf_score")
print("After drop:", df4.columns)

In [ ]:
# ── filter / where (identical) ───────────────────────────────────────────────
print("=== Filtering ===")

# Column expression
df.filter(F.col("salary") > 90000).select("name","salary").show()

# SQL string
df.filter("dept = 'Engineering' AND salary IS NOT NULL").select("name","dept","salary").show()

# Multiple conditions
df.filter(
    (F.col("dept") == "Engineering") &
    (F.col("salary") > 90000)
).show()

# isin
df.filter(F.col("dept").isin("Engineering", "HR")).show()

# NOT
df.filter(~F.col("dept").isin("Marketing")).show()

In [ ]:
# ── sort / orderBy ───────────────────────────────────────────────────────────
print("=== Sorting ===")

# Ascending (default)
df.orderBy("salary").select("name","salary").show(5)

# Descending
df.orderBy(F.col("salary").desc()).select("name","salary").show(5)

# Multiple columns — dept asc, salary desc within dept
df.orderBy(F.col("dept").asc(), F.col("salary").desc_nulls_last()).select("name","dept","salary").show()

# sort is an alias for orderBy
df.sort(F.desc("salary")).select("name","salary").show(3)

In [ ]:
# ── Aggregations ─────────────────────────────────────────────────────────────
print("=== Aggregations ===")

# groupBy + agg
df.groupBy("dept").agg(
    F.count("*").alias("count"),
    F.avg("salary").alias("avg_salary"),
    F.max("salary").alias("max_salary"),
    F.min("salary").alias("min_salary"),
    F.sum("salary").alias("total_salary"),
    F.countDistinct("perf_score").alias("distinct_scores"),
).orderBy("dept").show()

# count() on null — F.count("*") counts all rows, F.count("salary") skips nulls
print("Total rows         :", df.count())
print("Non-null salaries  :", df.select(F.count("salary")).collect()[0][0])
print("count(*) includes nulls — count(col) excludes nulls!")

In [ ]:
# ── Joins ────────────────────────────────────────────────────────────────────
print("=== Joins ===")

dept_info = spark.createDataFrame([
    ("Engineering", "San Francisco", 5_000_000),
    ("Marketing",   "New York",      2_000_000),
    ("Finance",     "Chicago",       1_500_000),  # no match in employees
], ["dept", "location", "budget"])

# INNER — only matching rows
inner = df.join(dept_info, on="dept", how="inner")
print(f"inner: {inner.count()} rows")
inner.select("name","dept","location").show()

# LEFT — all left rows, nulls for unmatched right
left = df.join(dept_info, on="dept", how="left")
print(f"left: {left.count()} rows (HR employees have null location)")

# ANTI — rows in left NOT in right
anti = df.join(dept_info, on="dept", how="left_anti")
print(f"anti (HR dept not in dept_info): {anti.count()} rows")
anti.select("name","dept").show()

# SEMI — rows in left that HAVE a match in right (no right cols added)
semi = df.join(dept_info, on="dept", how="left_semi")
print(f"semi: {semi.count()} rows — only left columns")
semi.select("name","dept").show()

In [ ]:
# ── Missing Data ─────────────────────────────────────────────────────────────
print("=== Missing Data ===")

# Count nulls per column
null_counts = df.select([
    F.count(F.when(F.col(c).isNull(), 1)).alias(c)
    for c in df.columns
])
print("Null counts:")
null_counts.show()

# dropna — drop rows with ANY null
print(f"Rows before dropna: {df.count()}")
print(f"Rows after dropna(any): {df.dropna().count()}")
print(f"Rows after dropna(subset=['salary']): {df.dropna(subset=['salary']).count()}")
print(f"Rows after dropna(thresh=6): {df.dropna(thresh=6).count()}")

# fillna — fill nulls with a value
df_filled = df.fillna({"salary": 0.0, "name": "Unknown"})
print("\nAfter fillna:")
df_filled.filter(F.col("id").isin(7, 9)).select("id","name","salary").show()

# replace
df_replaced = df.replace({"Engineering": "Eng", "Marketing": "Mkt"}, subset=["dept"])
df_replaced.select("dept").distinct().show()

In [ ]:
# ── union / unionByName ───────────────────────────────────────────────────────
print("=== Union ===")

df_a = df.filter(F.col("dept") == "Engineering")
df_b = df.filter(F.col("dept") == "HR")

# union — matches by POSITION (column order must match)
unioned = df_a.union(df_b)
print(f"union: {unioned.count()} rows")
unioned.show()

# unionByName — matches by COLUMN NAME (safer)
# Add an extra col to one df to show difference
df_b2 = df_b.withColumn("extra", F.lit("test"))
# This would fail: df_a.union(df_b2)
# unionByName with allowMissingColumns=True handles schema differences
unioned2 = df_a.unionByName(df_b2, allowMissingColumns=True)
print(f"unionByName (allowMissingColumns): {unioned2.count()} rows")
unioned2.select("name","dept","extra").show()

In [ ]:
# ── Repartition vs Coalesce ───────────────────────────────────────────────────
print("=== Repartition vs Coalesce ===")

print(f"Default partitions: {df.rdd.getNumPartitions()}")

# repartition — full shuffle, can increase OR decrease partitions
df_r = df.repartition(4)
print(f"After repartition(4): {df_r.rdd.getNumPartitions()} partitions")

# repartition by column — co-locate same dept on same partition
df_rc = df.repartition(4, "dept")
print(f"After repartition(4, 'dept'): {df_rc.rdd.getNumPartitions()} partitions")

# coalesce — reduce partitions WITHOUT full shuffle (more efficient for downsizing)
df_c = df.coalesce(2)
print(f"After coalesce(2): {df_c.rdd.getNumPartitions()} partitions")
print()
print("Rule of thumb:")
print("  repartition(N)  — when increasing OR when you need balanced partitions")
print("  coalesce(N)     — when DECREASING (no shuffle, faster, may be unbalanced)")
print("  repartition(col) — before writing to partition-by-column output")